In [11]:
import xarray as xr
import numpy as np
import glob

In [ ]:
start = 1979
end = 2020

In [8]:
landmask_5 = xr.open_dataset("../processed_data/gpcc_land_mask_5x5.nc")

In [30]:
### GPCC
ds = xr.open_dataset("../processed_data/gpcc/gpcc_mon_precip_5x5.nc")
ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})

ds_year.to_netcdf("../processed_data/gpcc/gpcc_year_precip_5x5.nc")

ds_year = ds_year.sel(year = slice(start, end))
ds_trend = ds_year.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/gpcc_trends/gpcc_annual_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [31]:
### GPCP

ds = xr.open_dataset("../processed_data/gpcp/gpcp_mon_precip_5x5.nc")
ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})

ds_year.to_netcdf("../processed_data/gpcp/gpcp_year_precip_5x5.nc")

ds_year = ds_year.sel(year = slice(start, end))
ds_trend = ds_year.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/gpcp_trends/gpcp_annual_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [32]:
### MSWEP

ds = xr.open_dataset("../processed_data/mswep/mswep_mon_precip_5x5.nc")
ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})

ds_year.to_netcdf("../processed_data/mswep/mswep_year_precip_5x5.nc")

ds_year = ds_year.sel(year = slice(start, end))
ds_trend = ds_year.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/mswep_trends/mswep_annual_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [33]:
### CPC
ds = xr.open_dataset("../processed_data/cpc/cpc_day_precip_5x5.nc")
ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})

ds_year.to_netcdf("../processed_data/cpc/cpc_year_precip_5x5.nc")

ds_year = ds_year.sel(year = slice(start, end))
ds_trend = ds_year.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/cpc_trends/cpc_annual_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [34]:
### MESACLIP

sim_keys = [".001.", ".002.", ".003.", ".004.", ".005.", ".006.", ".007.", ".009.", ".010"]

for sim in sim_keys:
    ds = xr.open_dataset("../processed_data/mesaclip/mesaclip_day_precip_"+sim.replace(".", "")+"_5x5.nc")
    ## convert to mm from m/s
    ds["pr"] = ds.pr*60*60*24*1000
    
    ds_mon = ds.resample(time = "ME").sum()
    ds_mon = xr.where(landmask_5 == 1, ds_mon.pr, np.nan).rename({"mask": "pr"})
    ds_mon.to_netcdf("../processed_data/mesaclip/mesaclip_mon_precip_"+sim.replace(".", "")+"_5x5.nc")
    
    ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
    ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})
    ds_year.to_netcdf("../processed_data/mesaclip/mesaclip_year_precip_"+sim.replace(".", "")+"_5x5.nc")

    ds_year = ds_year.sel(year = slice(start, end))
    ds_trend = ds_year.polyfit(dim = "year", deg = 1)
    ds_trend.to_netcdf("../processed_data/mesaclip_trends/mesaclip_annual_"+\
                       sim.replace(".", "")+"_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [ ]:
cmip_files = sorted(glob.glob("../processed_data/CMIP6/pr_mon*_5x5.nc"))

for f in cmip_files: 
    ds = xr.open_dataset(f)
    outfile = f.replace("mon", "year")
    ## convert to mm from kg/m2/s
    ds["pr"] = ds.pr*60*60*24*ds.time.dt.days_in_month
    
    ds_year = ds.groupby(ds.time.dt.year).sum(dim = "time")
    ds_year = xr.where(landmask_5 == 1, ds_year.pr, np.nan).rename({"mask": "pr"})
    
    ds_year.to_netcdf(outfile)

    ds_year = ds_year.sel(year = slice(start, end))
    ds_trend = ds_year.polyfit(dim = "year", deg = 1)
    ds_trend.to_netcdf("../processed_data/cmip_trends/"+\
                       f.split("/")[-1].replace("mon", "annual").replace(".nc", "_"+str(start)+"_"+str(end)+"_trend.nc"))